# test_2 — derniers tests
Tout est rechargé depuis `data/` : aucun téléchargement. À lancer depuis le dossier qui contient `data/`.

## 1. Rechargement des données

In [1]:
import os; os.environ["LOKY_MAX_CPU_COUNT"] = "4"   # coupe le warning joblib sous Windows
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("data/dataset.csv", index_col=0, parse_dates=True)
stocks = pd.read_csv("data/eia_stocks.csv", index_col=0, parse_dates=True).squeeze("columns")
intra = pd.read_csv("data/eia_intraday.csv", index_col=0, parse_dates=True)

lvl = stocks.shift(1).set_axis(stocks.index + pd.Timedelta(days=5))
e = pd.DataFrame({
    "r": 100 * df["r"],
    "S": 100 * df["surprise"] / lvl.reindex(df.index),
    "B": (df["slope"].shift(1) > 0).astype(int),
}).dropna()
e2 = e.join(intra, how="inner")
print(len(e), "événements journaliers,", len(e2), "intraday")

736 événements journaliers, 491 intraday


## 2. Probabilités du HMM, sans information future
HMM estimé sur 2007-2018, filtre de Hamilton, probabilité de la veille de chaque rapport. Nécessaire pour la partie 5.

In [2]:
from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler

feat = pd.read_csv("data/wti_hmm_features.csv", index_col=0, parse_dates=True).sort_index()
sc = StandardScaler().fit(feat[:"2018"])
X = sc.transform(feat)
hmm = GaussianHMM(3, covariance_type="full", n_iter=500, random_state=42).fit(X[feat.index <= "2018-12-31"])

ll = np.column_stack([multivariate_normal(hmm.means_[k], hmm.covars_[k]).logpdf(X) for k in range(3)])
a, p = np.zeros_like(ll), hmm.startprob_
for t in range(len(X)):                                  # filtre de Hamilton : P(S_t | infos jusqu'à t)
    w = p * np.exp(ll[t] - ll[t].max())
    a[t] = w / w.sum()
    p = a[t] @ hmm.transmat_
order = np.argsort(sc.inverse_transform(hmm.means_)[:, 1])          # tri par vol : 0 calme, 2 crise
P = pd.DataFrame(a[:, order], index=feat.index, columns=["P0", "P1", "P2"]).shift(1)   # veille de l'événement
e2 = e2.drop(columns=["P0", "P1", "P2"], errors="ignore").join(P.reindex(e2.index, method="ffill"))
print(e2[["P0", "P1", "P2"]].dropna().idxmax(axis=1).value_counts())

P1    231
P0    172
P2     88
Name: count, dtype: int64


## 3. L'effet de la courbe est-il réel ?
Chaque année a sa propre pente : S:B n'est estimé qu'à partir des années où le marché a changé d'état.
- S:B positif et significatif → effet réel
- S:B ≈ 0 → c'était la dérive dans le temps

In [3]:
m = smf.ols("r5 ~ S * C(an) + S:B + B", data=e2.assign(an=e2.index.year)).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(pd.DataFrame({"coef": m.params, "p": m.pvalues}).loc[["S:B", "B"]].round(3), "\n")
print(pd.crosstab(e2.index.year.to_series(index=e2.index, name="an"), e2["B"]))   # nb de rapports détendus (0) / tendus (1) par année

      coef      p
S:B  0.056  0.172
B   -0.066  0.274 

B      0   1
an          
2013  27  13
2014  10  30
2015  46   0
2016  45   0
2017  44   0
2018  11  31
2019  30  14
2020  43   3
2021  10  36
2022   4  40
2023  20  23
2024   2   9


## 4. Diagnostic de la stratégie (entrée 10h35, sortie 10h45)
- Pente apprise > 0 → pari sur un retour en arrière ; < 0 → pari que le mouvement continue
- Gain réparti sur les années → solide ; concentré sur 1-2 années → fragile

In [4]:
COST = 0.04
d = e2.assign(y=e2["r15"] - e2["r5"])
rows = {}
for yr in range(2016, 2025):
    tr, te = d.loc[str(yr - 3):str(yr - 1)], d.loc[str(yr)]
    m = smf.ols("y ~ S", data=tr).fit()
    pnl = np.sign(m.predict(te)) * te["y"] - COST
    rows[yr] = {"pente apprise": m.params["S"], "trades": len(pnl), "gain net total (%)": pnl.sum(), "taux de réussite": (pnl > 0).mean()}
pd.DataFrame(rows).T.round(3)

,pente apprise,trades,gain net total (%),taux de réussite
2016,0.035,45.0,11.520,0.578
2017,0.039,44.0,4.680,0.523
2018,0.076,42.0,-3.288,0.429
2019,0.051,44.0,4.326,0.636
2020,0.046,46.0,11.455,0.500
2021,0.106,46.0,-2.835,0.435
2022,0.103,44.0,1.185,0.545
2023,0.100,43.0,4.141,0.628
2024,0.031,11.0,-1.996,0.273


## 5. Stratégie : base vs courbe vs HMM
Mêmes règles, même échantillon, fenêtre glissante de 3 ans, coût 0,04 % par aller-retour.
`t-stat écart vs base` > 2 → le régime améliore significativement la stratégie.

In [5]:
COST = 0.04
models = {"base": "y ~ S", "+ courbe (B)": "y ~ S * B", "+ HMM": "y ~ S * (P1 + P2)"}
rows = {}
for h in [15, 30, 60]:
    d = e2.dropna(subset=["P1"]).assign(y=lambda x: x[f"r{h}"] - x["r5"])   # de 10h35 à la sortie
    pnl = {k: [] for k in models}
    for yr in range(2016, 2025):
        tr, te = d.loc[str(yr - 3):str(yr - 1)], d.loc[str(yr)]
        for k, f in models.items():
            pnl[k].append(np.sign(smf.ols(f, data=tr).fit().predict(te)) * te["y"] - COST)
    pnl = {k: pd.concat(v) for k, v in pnl.items()}
    ans = (pnl["base"].index[-1] - pnl["base"].index[0]).days / 365.25
    for k, p in pnl.items():
        diff = p - pnl["base"]                                                  # gain en plus (ou en moins) vs base
        rows[(f"+{h} min", k)] = {"Sharpe net": p.mean() / p.std() * np.sqrt(len(p) / ans),
                                  "t-stat": p.mean() / p.std() * np.sqrt(len(p)),
                                  "t-stat écart vs base": np.nan if k == "base" else diff.mean() / diff.std() * np.sqrt(len(diff))}
pd.DataFrame(rows).T.round(2)

Sharpe net  t-stat  t-stat écart vs base
+15 min base                0.71    2.05                   NaN
        + courbe (B)        0.29    0.85                 -2.21
        + HMM               0.40    1.15                 -1.21
+30 min base                0.39    1.12                   NaN
        + courbe (B)        0.30    0.87                 -1.75
        + HMM               0.02    0.06                 -1.38
+60 min base               -0.07   -0.20                   NaN
        + courbe (B)       -0.00   -0.01                  0.17
        + HMM               0.13    0.36                  0.45

## 6. Test final : mai 2024 → aujourd'hui (données jamais utilisées)
Règle **figée avant de regarder** : entrée 10h35, sortie 10h45, `y ~ S` estimé sur les 3 années civiles précédentes, coût 0,04 %.
Ne rien modifier après avoir vu le résultat, sinon ce n'est plus un test.

Utilise uniquement les stocks EIA et les prix minute Dukascopy (gratuits, à jour). Premier lancement ~3 min, ensuite en cache dans `data/eia_intraday_recent.csv`.

In [6]:
# TEST FINAL sur des données jamais utilisées : mai 2024 -> aujourd'hui
# Règle figée avant de regarder : entrée 10h35, sortie 10h45, y ~ S estimé sur les 3 années civiles précédentes, coût 0,04 %
import os, dukascopy_python as dk
from dukascopy_python.instruments import INSTRUMENT_CMD_ENERGY_E_LIGHT as WTI_CFD
from pandas.tseries.holiday import USFederalHolidayCalendar

COST, H, CUT = 0.04, [5, 15, 30, 60], "2024-04-06"

# 1. Stocks EIA à jour + surprise (même méthode qu'avant)
url = "https://www.eia.gov/dnav/pet/hist_xls/WCESTUS1w.xls"
st = pd.read_excel(url, sheet_name="Data 1", skiprows=2, index_col=0,
                   storage_options={"User-Agent": "Mozilla/5.0"}).squeeze("columns") / 1e3
dd = st.diff()
seas = dd.groupby(dd.index.isocalendar().week).transform(lambda s: s.shift(1).rolling(5).mean())
S_all = (100 * (dd - seas) / st.shift(1)).dropna().rename("S")
S_all.index = S_all.index + pd.Timedelta(days=5)              # date de publication (mercredi)

# 2. Prix minute Dukascopy des nouveaux mercredis (téléchargés une seule fois, ~3 min)
fer = USFederalHolidayCalendar().holidays(CUT, pd.Timestamp.today())
pub = [d for d in S_all.loc[CUT:].index if not ((fer >= d - pd.Timedelta(days=2)) & (fer <= d)).any()]

def window(d):
    t0 = pd.Timestamp(f"{d:%Y-%m-%d} 10:25", tz="America/New_York")
    b = dk.fetch(WTI_CFD, dk.INTERVAL_MIN_1, dk.OFFER_SIDE_BID, t0, t0 + pd.Timedelta(minutes=70), limit=100)
    return b["close"].tz_convert("America/New_York").tz_localize(None).asfreq("1min").ffill()

f = "data/eia_intraday_recent.csv"
if os.path.exists(f):
    intra_new = pd.read_csv(f, index_col=0, parse_dates=True)
else:
    rows = {}
    for d in pub:
        p, t = window(d), d + pd.Timedelta("10:29:00")
        if t in p.index and t + pd.Timedelta(minutes=60) in p.index:
            rows[d] = {f"r{h}": 100 * np.log(p[t + pd.Timedelta(minutes=h)] / p[t]) for h in H}
    intra_new = pd.DataFrame.from_dict(rows, orient="index")
    intra_new.to_csv(f)

# 3. Stratégie figée, évaluée UNIQUEMENT sur les nouveaux rapports
ev_all = pd.concat([intra, intra_new]).join(S_all, how="inner")
ev_all["y"] = ev_all["r15"] - ev_all["r5"]                  # de 10h35 à 10h45
pnl = []
for yr in range(2024, pd.Timestamp.today().year + 1):
    tr, te = ev_all.loc[str(yr - 3):str(yr - 1)], ev_all.loc[str(yr)].loc[CUT:]
    pnl.append(np.sign(smf.ols("y ~ S", data=tr).fit().predict(te)) * te["y"] - COST)
p = pd.concat(pnl)
ans = (p.index[-1] - p.index[0]).days / 365.25
print(f"{len(p)} trades jamais vus, du {p.index[0]:%d/%m/%Y} au {p.index[-1]:%d/%m/%Y}\n")
print(pd.Series({"gain moyen net (%)": p.mean(), "taux de réussite": (p > 0).mean(),
                 "t-stat": p.mean() / p.std() * np.sqrt(len(p)),
                 "Sharpe annuel net": p.mean() / p.std() * np.sqrt(len(p) / ans),
                 "Sharpe annuel brut": (p.mean() + COST) / p.std() * np.sqrt(len(p) / ans)}).round(3), "\n")
print(p.groupby(p.index.year).agg(["count", "sum"]).rename(columns={"count": "trades", "sum": "gain net total (%)"}).round(2))

115 trades jamais vus, du 10/04/2024 au 21/10/2026

gain moyen net (%)   -0.060
taux de réussite      0.435
t-stat               -2.082
Sharpe annuel net    -1.309
Sharpe annuel brut   -0.439
dtype: float64 

      trades  gain net total (%)
2024      32               -1.05
2025      46               -4.36
2026      37               -1.51


In [7]:
print(smf.ols("r5 ~ S", data=ev_all.loc[CUT:]).fit(cov_type="HAC", cov_kwds={"maxlags": 4}).summary().tables[1])

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0056      0.034     -0.164      0.870      -0.072       0.061
S             -0.0493      0.019     -2.560      0.010      -0.087      -0.012
